In [1]:
import os
os.chdir(r"C:\vscode\graph-rag\Source")

In [2]:
%pwd

'C:\\vscode\\graph-rag\\Source'

In [22]:
from config.settings_loader import load_config

config = load_config("config/config.yaml")

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import json

chunks = []

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=config["chunking"]["chunk_size"],
    chunk_overlap=config["chunking"]["chunk_overlap"],
    length_function=len,
    separators=[". ", "."]
)

# Process both volumes
volumes = [
    ("volume_1", config["data_source"]["jsonified_data"]["volume_1"]),
    ("volume_2", config["data_source"]["jsonified_data"]["volume_2"])
]

for volume_name, file_path in volumes:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Convert to Documents and split
    documents = [
        Document(page_content=entry['text'], metadata=entry['metadata'])
        for entry in data
    ]
    
    volume_chunks = text_splitter.split_documents(documents)
    chunks.extend(volume_chunks)
    
    print(f"Original entries for {volume_name}: {len(data)}")
    print(f"New chunks for {volume_name}: {len(volume_chunks)}")

print(f"\nTotal chunks: {len(chunks)}")

Original entries for volume_1: 132
New chunks for volume_1: 1096
Original entries for volume_2: 150
New chunks for volume_2: 1374

Total chunks: 2470


In [10]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
import os

load_dotenv()

# Initialize embeddings
embeddings = OpenAIEmbeddings(
    model=config["embedding"]["embedding_model"],
    api_key=os.getenv("OPENAI_API_KEY")
)

# Create FAISS vector store from chunks
vector_store = FAISS.from_documents(chunks, embeddings)

# Save to disk
vector_store_path = config["embedding"]["vector_store_path"]
vector_store.save_local("C:/vscode/graph-rag/Source/data/vector_store")

print(f"Vector store created and saved to {vector_store_path}")
print(f"Total vectors stored: {len(chunks)}")

Vector store created and saved to dataector_store
Total vectors stored: 2470


In [ ]:
# Load the vector store from disk
loaded_vector_store = FAISS.load_local(
    config["embedding"]["vector_store_path"],  # Use the same path you used for saving
    embeddings,
    allow_dangerous_deserialization=True
)

# Create a retriever
retriever = loaded_vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5}
)

data/vector_store


In [ ]:
# Invoke the retriever
query = "How does the author argue that Roman persecution of early Christians was driven more by concerns for public order, political stability, and social conformity than by consistent religious hatred, and what evidence does he use to support this interpretation?"
retrieved_docs = retriever.invoke(query)

print(f"Query: {query}\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Document {i} ---")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}\n")

Query: How does the author argue that Roman persecution of early Christians was driven more by concerns for public order, political stability, and social conformity than by consistent religious hatred, and what evidence does he use to support this interpretation?

--- Document 1 ---
Content: . If, on the contrary, they failed in their proofs, they incurred the severe and perhaps capital penalty, which, according to a law published by the emperor Hadrian, was inflicted on those who falsely attributed to their fellow-citizens the crime of Christianity. The violence of personal or superstitious animosity might sometimes prevail over the most natural apprehensions of disgrace and danger but it cannot surely be imagined, that accusations of so unpromising an appearance were either lightly or frequently undertaken by the Pagan subjects of the Roman empire. The expedient which was employed to elude the prudence of the laws, affords a sufficient proof how effectually they disappointed the misc